# Replicate Consistency Analysis

Before building the model, we need to understand how consistent the 3 replicates are.

**Key questions:**
1. How often do replicates agree (all zero, all nonzero, or mixed)?
2. What's the variation across replicates when there IS signal?
3. Are there outlier replicates that should be excluded?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

# Load raw data with replicates
df_amines = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_amines_for_heatmap_manual.csv")
df_subs = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_subs_for_heatmap_manual.csv")

print(f"Amines table: {df_amines.shape}")
print(f"Subs table: {df_subs.shape}")
print(f"\nReplicates: {df_amines['Replicate'].unique()}")

In [ ]:
# Combine both tables and melt to long format
def melt_to_long(df):
    meta_cols = ['filename', 'Code', 'Replicate']
    product_cols = [c for c in df.columns if c not in meta_cols]
    
    long = df.melt(
        id_vars=meta_cols,
        value_vars=product_cols,
        var_name='ProductName',
        value_name='Intensity'
    )
    return long

long_amines = melt_to_long(df_amines)
long_subs = melt_to_long(df_subs)
long_all = pd.concat([long_amines, long_subs], ignore_index=True)

# Rename Code to Enzyme
long_all = long_all.rename(columns={'Code': 'Enzyme'})

print(f"Total measurements: {len(long_all):,}")
print(f"Unique enzymes: {long_all['Enzyme'].nunique()}")
print(f"Unique products: {long_all['ProductName'].nunique()}")
long_all.head()

In [ ]:
# For each (Enzyme, Product) pair, look at the 3 replicates
replicate_stats = long_all.groupby(['Enzyme', 'ProductName']).agg(
    rep1=('Intensity', lambda x: x.iloc[0] if len(x) > 0 else np.nan),
    rep2=('Intensity', lambda x: x.iloc[1] if len(x) > 1 else np.nan),
    rep3=('Intensity', lambda x: x.iloc[2] if len(x) > 2 else np.nan),
    n_reps=('Intensity', 'count'),
    mean_intensity=('Intensity', 'mean'),
    std_intensity=('Intensity', 'std'),
    max_intensity=('Intensity', 'max'),
    min_intensity=('Intensity', 'min'),
    median_intensity=('Intensity', 'median')
).reset_index()

print(f"Unique (Enzyme, Product) pairs: {len(replicate_stats):,}")
print(f"\nReplicate counts:")
print(replicate_stats['n_reps'].value_counts())

In [ ]:
# Classify replicate agreement
def classify_replicates(row):
    reps = [row['rep1'], row['rep2'], row['rep3']]
    reps = [r for r in reps if pd.notna(r)]  # Remove NaN
    
    n_zero = sum(1 for r in reps if r == 0)
    n_nonzero = sum(1 for r in reps if r > 0)
    
    if n_zero == len(reps):
        return 'all_zero'
    elif n_nonzero == len(reps):
        return 'all_nonzero'
    else:
        return 'mixed'

replicate_stats['agreement'] = replicate_stats.apply(classify_replicates, axis=1)

print("=== REPLICATE AGREEMENT ===")
agreement_counts = replicate_stats['agreement'].value_counts()
print(agreement_counts)
print(f"\nPercentages:")
print((agreement_counts / len(replicate_stats) * 100).round(1))

In [ ]:
# Focus on MIXED cases - these are problematic
mixed = replicate_stats[replicate_stats['agreement'] == 'mixed'].copy()

print(f"=== MIXED REPLICATES (DISAGREEMENT) ===")
print(f"Total mixed cases: {len(mixed):,} ({100*len(mixed)/len(replicate_stats):.1f}%)")

# How many zeros vs nonzeros in mixed cases?
def count_zeros(row):
    reps = [row['rep1'], row['rep2'], row['rep3']]
    reps = [r for r in reps if pd.notna(r)]
    return sum(1 for r in reps if r == 0)

mixed['n_zeros'] = mixed.apply(count_zeros, axis=1)

print(f"\nIn mixed cases, how many replicates are zero?")
print(mixed['n_zeros'].value_counts().sort_index())

In [ ]:
# Look at specific examples of mixed replicates
print("=== EXAMPLES OF MIXED REPLICATES ===")
print("\nCases where 2 reps are zero, 1 has signal:")
examples_2zero = mixed[mixed['n_zeros'] == 2].head(10)
print(examples_2zero[['Enzyme', 'ProductName', 'rep1', 'rep2', 'rep3', 'max_intensity']])

print("\nCases where 1 rep is zero, 2 have signal:")
examples_1zero = mixed[mixed['n_zeros'] == 1].head(10)
print(examples_1zero[['Enzyme', 'ProductName', 'rep1', 'rep2', 'rep3', 'max_intensity']])

In [ ]:
# For all_nonzero cases, what's the coefficient of variation?
nonzero = replicate_stats[replicate_stats['agreement'] == 'all_nonzero'].copy()

# CV = std / mean (only meaningful when mean > 0)
nonzero['cv'] = nonzero['std_intensity'] / nonzero['mean_intensity']

print("=== VARIATION IN ALL-NONZERO CASES ===")
print(f"Total all-nonzero cases: {len(nonzero):,}")
print(f"\nCoefficient of Variation (CV) statistics:")
print(nonzero['cv'].describe())

# Plot CV distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(nonzero['cv'].dropna(), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Coefficient of Variation (CV)')
axes[0].set_ylabel('Count')
axes[0].set_title('CV Distribution (all-nonzero replicates)')
axes[0].axvline(0.5, color='red', linestyle='--', label='CV=0.5 (50% variation)')
axes[0].axvline(1.0, color='orange', linestyle='--', label='CV=1.0 (100% variation)')
axes[0].legend()

# CV vs mean intensity
axes[1].scatter(np.log10(nonzero['mean_intensity'] + 1), nonzero['cv'], alpha=0.3, s=10)
axes[1].set_xlabel('log10(Mean Intensity)')
axes[1].set_ylabel('CV')
axes[1].set_title('CV vs Mean Intensity')
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cv_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nCases with CV > 1.0 (very high variation): {(nonzero['cv'] > 1.0).sum()}")
print(f"Cases with CV > 0.5 (moderate variation): {(nonzero['cv'] > 0.5).sum()}")

## High-CV Outlier Analysis

Identify specific enzyme-product pairs with CV > 1.0 (standard deviation exceeds the mean). These represent measurements where replicates wildly disagree — possibly due to technical failures, contamination, or carryover.

In [ ]:
# ── High-CV outliers: CV > 1.0 ──
high_cv = nonzero[nonzero['cv'] > 1.0].sort_values('cv', ascending=False).copy()

print(f"=== HIGH-CV OUTLIERS (CV > 1.0): {len(high_cv)} cases ===\n")

# Parse Enzyme and ProductName to extract amine info
# ProductName format is like "EnzymeName_Amine_BileAcid" — let's just show the raw columns
display_cols = ['Enzyme', 'ProductName', 'rep1', 'rep2', 'rep3', 
                'mean_intensity', 'std_intensity', 'cv']

print("Top 30 most variable measurements:")
print(high_cv[display_cols].head(30).to_string(index=False))

print(f"\n\n--- Summary of high-CV cases ---")
print(f"Total high-CV (CV > 1.0): {len(high_cv)}")
print(f"Mean intensity range: {high_cv['mean_intensity'].min():.0f} – {high_cv['mean_intensity'].max():.0f}")
print(f"Max replicate range: {high_cv['max_intensity'].min():.0f} – {high_cv['max_intensity'].max():.0f}")

# How many of these have high mean intensity (not just noise)?
high_cv_high_signal = high_cv[high_cv['mean_intensity'] > 1000]
print(f"\nHigh-CV AND high mean intensity (>1000): {len(high_cv_high_signal)}")
print("These are the MOST suspicious — strong signal in some reps, weak/none in others:")
if len(high_cv_high_signal) > 0:
    print(high_cv_high_signal[display_cols].to_string(index=False))

In [ ]:
# ── Which ENZYMES are repeat offenders for high CV? ──
print("=== ENZYMES WITH MOST HIGH-CV CASES (CV > 1.0) ===\n")
enzyme_highcv = high_cv.groupby('Enzyme').agg(
    n_high_cv=('cv', 'size'),
    mean_cv=('cv', 'mean'),
    max_cv=('cv', 'max'),
    n_high_signal=('mean_intensity', lambda x: (x > 1000).sum())
).sort_values('n_high_cv', ascending=False)

print(enzyme_highcv.head(20).to_string())

print(f"\n\n=== PRODUCTS WITH MOST HIGH-CV CASES ===\n")
product_highcv = high_cv.groupby('ProductName').agg(
    n_high_cv=('cv', 'size'),
    mean_cv=('cv', 'mean'),
    max_cv=('cv', 'max'),
).sort_values('n_high_cv', ascending=False)

print(product_highcv.head(20).to_string())

In [ ]:
# ── Visualize high-CV outliers ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Bar chart: top 15 enzymes by high-CV count
top_enz = enzyme_highcv.head(15)
colors_enz = ['#d62728' if n > 0 else 'steelblue' for n in top_enz['n_high_signal']]
axes[0].barh(range(len(top_enz)), top_enz['n_high_cv'], color=colors_enz, alpha=0.8)
axes[0].set_yticks(range(len(top_enz)))
axes[0].set_yticklabels(top_enz.index, fontsize=8)
axes[0].set_xlabel('Number of high-CV cases')
axes[0].set_title('Top 15 Enzymes by High-CV Count\n(red = has high-signal outliers)')
axes[0].invert_yaxis()

# 2. Bar chart: top 15 products by high-CV count
top_prod = product_highcv.head(15)
axes[1].barh(range(len(top_prod)), top_prod['n_high_cv'], color='darkorange', alpha=0.8)
axes[1].set_yticks(range(len(top_prod)))
axes[1].set_yticklabels(top_prod.index, fontsize=7)
axes[1].set_xlabel('Number of high-CV cases')
axes[1].set_title('Top 15 Products by High-CV Count')
axes[1].invert_yaxis()

# 3. Scatter: for high-CV cases, show max vs min replicate
axes[2].scatter(np.log10(high_cv['min_intensity'] + 1), 
                np.log10(high_cv['max_intensity'] + 1), 
                c=high_cv['cv'], cmap='Reds', alpha=0.6, s=20, edgecolors='k', linewidths=0.3)
axes[2].plot([0, 8], [0, 8], 'k--', alpha=0.3, label='perfect agreement')
axes[2].set_xlabel('log10(Min Replicate + 1)')
axes[2].set_ylabel('log10(Max Replicate + 1)')
axes[2].set_title('Max vs Min Replicate (high-CV cases)\nColor = CV')
cb = plt.colorbar(axes[2].collections[0], ax=axes[2], label='CV')
axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'high_cv_outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {OUTPUT_DIR / 'high_cv_outlier_analysis.png'}")

In [ ]:
# ── Save high-CV outliers to CSV ──
high_cv_save = high_cv[['Enzyme', 'ProductName', 'rep1', 'rep2', 'rep3', 
                         'mean_intensity', 'std_intensity', 'cv', 
                         'max_intensity', 'min_intensity']].copy()
high_cv_save = high_cv_save.sort_values('cv', ascending=False)
high_cv_save.to_csv(OUTPUT_DIR / 'high_cv_outliers.csv', index=False)

print(f"Saved {len(high_cv_save)} high-CV outlier cases to: {OUTPUT_DIR / 'high_cv_outliers.csv'}")
print(f"\nBreakdown by CV range:")
print(f"  CV 1.0–1.25: {((high_cv_save['cv'] >= 1.0) & (high_cv_save['cv'] < 1.25)).sum()}")
print(f"  CV 1.25–1.5: {((high_cv_save['cv'] >= 1.25) & (high_cv_save['cv'] < 1.5)).sum()}")
print(f"  CV 1.5–1.75: {((high_cv_save['cv'] >= 1.5) & (high_cv_save['cv'] < 1.75)).sum()}")
print(f"  CV > 1.75:   {(high_cv_save['cv'] >= 1.75).sum()}")

In [ ]:
# Which enzymes have the most inconsistent replicates?
enzyme_mixed = mixed.groupby('Enzyme').size().sort_values(ascending=False)

print("=== ENZYMES WITH MOST MIXED REPLICATES ===")
print("(These enzymes have inconsistent measurements)")
print(enzyme_mixed.head(20))

In [ ]:
# Which products have the most inconsistent replicates?
product_mixed = mixed.groupby('ProductName').size().sort_values(ascending=False)

print("=== PRODUCTS WITH MOST MIXED REPLICATES ===")
print("(These products have inconsistent measurements)")
print(product_mixed.head(20))

In [ ]:
# Check if certain replicates are systematically different
print("=== REPLICATE-LEVEL STATISTICS ===")

# Mean intensity per replicate
rep_means = long_all.groupby('Replicate')['Intensity'].agg(['mean', 'median', 'std', 'sum'])
print("\nOverall intensity by replicate:")
print(rep_means)

# Non-zero counts per replicate
rep_nonzero = long_all.groupby('Replicate').apply(lambda x: (x['Intensity'] > 0).sum())
print(f"\nNon-zero measurements per replicate:")
print(rep_nonzero)

In [ ]:
# Pairwise replicate correlation
# Pivot to get rep1, rep2, rep3 as columns
pivot = long_all.pivot_table(
    index=['Enzyme', 'ProductName'],
    columns='Replicate',
    values='Intensity',
    aggfunc='first'
).reset_index()

print("=== PAIRWISE REPLICATE CORRELATIONS ===")
if 'rep1' in pivot.columns and 'rep2' in pivot.columns:
    corr_12 = pivot['rep1'].corr(pivot['rep2'])
    corr_13 = pivot['rep1'].corr(pivot['rep3'])
    corr_23 = pivot['rep2'].corr(pivot['rep3'])
    
    print(f"Correlation rep1 vs rep2: {corr_12:.3f}")
    print(f"Correlation rep1 vs rep3: {corr_13:.3f}")
    print(f"Correlation rep2 vs rep3: {corr_23:.3f}")

In [ ]:
# Visualize replicate correlations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Filter to non-zero for better visualization
mask = (pivot['rep1'] > 0) | (pivot['rep2'] > 0) | (pivot['rep3'] > 0)
pv = pivot[mask].copy()

for ax, (r1, r2), title in zip(axes, 
                                [('rep1', 'rep2'), ('rep1', 'rep3'), ('rep2', 'rep3')],
                                ['Rep1 vs Rep2', 'Rep1 vs Rep3', 'Rep2 vs Rep3']):
    ax.scatter(np.log10(pv[r1] + 1), np.log10(pv[r2] + 1), alpha=0.2, s=5)
    ax.plot([0, 8], [0, 8], 'r--', label='y=x')
    ax.set_xlabel(f'log10({r1} + 1)')
    ax.set_ylabel(f'log10({r2} + 1)')
    ax.set_title(title)
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'replicate_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary and recommendations
print("=" * 60)
print("SUMMARY & RECOMMENDATIONS")
print("=" * 60)

total = len(replicate_stats)
n_all_zero = (replicate_stats['agreement'] == 'all_zero').sum()
n_all_nonzero = (replicate_stats['agreement'] == 'all_nonzero').sum()
n_mixed = (replicate_stats['agreement'] == 'mixed').sum()

print(f"\nTotal (Enzyme, Product) pairs: {total:,}")
print(f"\nReplicate agreement:")
print(f"  All zero:    {n_all_zero:,} ({100*n_all_zero/total:.1f}%) - clearly inactive")
print(f"  All nonzero: {n_all_nonzero:,} ({100*n_all_nonzero/total:.1f}%) - clearly active")
print(f"  Mixed:       {n_mixed:,} ({100*n_mixed/total:.1f}%) - UNCERTAIN")

print(f"\n=== FOR MODEL TRAINING ===")
print(f"\nOption 1: Use median (current approach)")
print(f"  - Simple, robust to outliers")
print(f"  - Loses information about uncertainty")

print(f"\nOption 2: Require 2/3 replicates to agree")
print(f"  - More confident labels")
print(f"  - Excludes {n_mixed:,} mixed cases")

print(f"\nOption 3: Use all replicates as separate training examples")
print(f"  - 3x more data")
print(f"  - But replicates are not independent")

print(f"\nOption 4: Weight by replicate consistency")
print(f"  - Give higher weight to consistent measurements")
print(f"  - More complex but principled")

In [ ]:
# Save replicate stats for later use
replicate_stats.to_csv(OUTPUT_DIR / 'replicate_consistency_stats.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'replicate_consistency_stats.csv'}")